# Governance Mark Emails Sent

Runs as the **final stage** of the pipeline, after all Outlook activities have completed.

Marks every `pending` row in `email_outbox` as `sent` so the next pipeline run doesn't resend them.

### Why this is needed
The pipeline's Lookup activities filter on `status = 'pending'`. Without this step, every run
would resend all previously generated emails.

### Permission
**Contributor** — writes to a Delta table only. No API calls.


## Parameters

In [ ]:
# Set by the pipeline (or leave default for manual runs)
mark_status = "sent"   # or "failed" if you wire a failure path


## Mark pending emails

In [ ]:
import pandas as pd
import pyodbc, struct, notebookutils
import logging
from datetime import datetime, timezone

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("mark_sent")

run_timestamp = datetime.now(timezone.utc).isoformat()

# Warehouse connection (Governance schema in DW_Fabric)
WAREHOUSE_SQL_ENDPOINT = "<WAREHOUSE_SQL_ENDPOINT>"
WAREHOUSE_DATABASE     = "DW_Fabric"
GOVERNANCE_SCHEMA      = "Governance"

def get_warehouse_connection():
    wh_token = notebookutils.credentials.getToken("https://database.windows.net/")
    token_bytes = wh_token.encode("utf-16-le")
    token_struct = struct.pack(f'<I{len(token_bytes)}s', len(token_bytes), token_bytes)
    SQL_COPT_SS_ACCESS_TOKEN = 1256
    conn_str = (
        f"Driver={{ODBC Driver 18 for SQL Server}};"
        f"Server={WAREHOUSE_SQL_ENDPOINT};"
        f"Database={WAREHOUSE_DATABASE};"
        f"Encrypt=Yes;TrustServerCertificate=No"
    )
    return pyodbc.connect(conn_str, attrs_before={SQL_COPT_SS_ACCESS_TOKEN: token_struct})

conn = get_warehouse_connection()

# Warehouse tables can be updated in place — no need to read the whole outbox into pandas,
# flip the pending rows, and write the whole table back like the Delta version did.
cursor = conn.cursor()
cursor.execute(
    f"UPDATE {GOVERNANCE_SCHEMA}.email_outbox SET status = ?, sent_at = ? WHERE status = 'pending'",
    mark_status, run_timestamp
)
pending_count = cursor.rowcount
conn.commit()

if pending_count <= 0:
    log.info("No pending emails. Nothing to mark.")
else:
    log.info(f"✔ Marked {pending_count} emails as '{mark_status}'")


## Summary

In [ ]:
df_final = pd.read_sql(f"""
    SELECT status, email_type, COUNT(*) as count
    FROM {GOVERNANCE_SCHEMA}.email_outbox
    GROUP BY status, email_type
    ORDER BY status, email_type
""", conn)

print("=" * 55)
print("  EMAIL OUTBOX STATE")
print("=" * 55)
if df_final.empty:
    print("  Outbox is empty.")
else:
    for _, r in df_final.iterrows():
        print(f"  {r['status']:10s} {r['email_type']:28s} {r['count']}")
    print(f"  {'-' * 51}")
    print(f"  {'Total':10s} {'':28s} {df_final['count'].sum()}")
print("=" * 55)


## Purge old sent emails

The outbox grows with every run. Each row holds a full HTML body (~20 KB), so it is worth
trimming periodically. Runs automatically now, every pipeline run — keeps only the last 30 days.


In [ ]:
from datetime import timedelta

cutoff = (datetime.now(timezone.utc) - timedelta(days=30)).isoformat()

cursor = conn.cursor()
cursor.execute(f"DELETE FROM {GOVERNANCE_SCHEMA}.email_outbox WHERE created_at < ?", cutoff)
purged = cursor.rowcount
conn.commit()

print(f"Purged {purged} rows older than 30 days.")
